# Using DataLad run to Record Provenance

**Author**: Sin Kim, Michèle Masson-Trottier

<div style="line-height: 2;">
<a href="https://github.com/kimsin98"><img src="https://img.shields.io/badge/-Sin_Kim-181717?logo=github" alt="GitHub"></a><br>
<a href="https://github.com/micmas"><img src="https://img.shields.io/badge/-Michèle_Masson--Trottier-181717?logo=github" alt="GitHub"></a> <a href="https://orcid.org/0000-0002-0642-5662"><img src="https://img.shields.io/badge/ORCID-0000--0002--0642--5662-green?logo=orcid" alt="ORCID"></a>
</div>

**Date**: 30/03/2026

**License:** 
<div style="margin-top: 10px;">
    <a href="https://creativecommons.org/licenses/by/4.0/" target="_blank" style="color: #0066cc;">
        <i class="fas fa-balance-scale"></i> CC-BY-4.0 License
    </a>
</div>

## Purpose

Using `datalad run` to record how analysis results were produced, enabling others to re-run analyses reproducibly.

## Learning Objectives

- Create a DataLad dataset configured for analysis projects (YODA)
- Write and track analysis scripts within DataLad
- Use `datalad run` to record the provenance of result files
- View Git commit history and re-run analyses with `datalad rerun`

## Citations

- **DataLad** — Halchenko, Y., et al. (2021). DataLad: distributed system for joint management of code, data, and their relationship. *Journal of Open Source Software*, 6(63), 3262. https://doi.org/10.21105/joss.03262

### Educational Resources

- [DataLad Handbook](http://handbook.datalad.org/en/latest/)
- [YODA principles](http://handbook.datalad.org/en/latest/basics/101-127-yoda.html)
- [Brief terminal guide](http://handbook.datalad.org/en/latest/intro/howto.html)
- [datalad-container extension](http://handbook.datalad.org/en/latest/basics/101-133-containersrun.html)

## Prerequisites

- Running Neurodesk
- Familiarity with terminal navigation
- Basic Git knowledge (helpful)

## Section 1: Create a DataLad Project

Open a terminal and navigate to Neurodesktop storage:

In [ ]:
cd ~/neurodesktop-storage/

datalad create -c yoda SomeProject

Expected output:

```
[INFO   ] Creating a new annex repo at /home/jovyan/neurodesktop-storage/SomeProject
create(ok): /home/jovyan/neurodesktop-storage/SomeProject (dataset)
```

> **What is YODA?** The `-c yoda` option configures the dataset according to [YODA principles](http://handbook.datalad.org/en/latest/basics/101-127-yoda.html) — a set of organisational principles for reproducible data analyses that works well with version control. It pre-creates `code/`, `README.md`, and `CHANGELOG.md`.

Check the dataset contents:

In [ ]:
cd SomeProject
ls

Output:

```
CHANGELOG.md  README.md  code
```

## Section 2: Create a Script

Load Julia and create a simple script:

In [ ]:
ml julia

cat > code/hello.jl << 'EOF'
println("hello neurodesktop")
EOF

> **About heredoc (`<< EOF`)** — This creates the script file using Bash terminal commands only. You can also use any text editor (e.g., nano, VSCode) to create `code/hello.jl`.

Test the script:

In [ ]:
julia code/hello.jl > hello.txt
cat hello.txt

Output:

```
hello neurodesktop
```

## Section 3: Run and Record with DataLad

Check what has changed:

In [ ]:
datalad status

Output:

```
untracked: .../code/hello.jl (file)
untracked: .../hello.txt (file)
```

Save the script and clean the test output:

In [ ]:
datalad save -m 'hello script' code/

git clean -i

Select option 1 (clean) to remove `hello.txt`.

> **git clean vs git reset** — `git clean` removes new, untracked files. To reset existing modified files to the last saved version, use `git reset --hard`.

Create the output directory and use `datalad run`:

In [ ]:
mkdir outputs

datalad run -m 'run hello' -o 'outputs/hello.txt' 'julia code/hello.jl > outputs/hello.txt'

**Arguments explained:**

- `-m 'run hello'` — human-readable message recorded in the dataset log
- `-o 'outputs/hello.txt'` — expected output (supports multiple `-o` and wildcards)
- The final argument is the command DataLad will run and record

Verify:

In [ ]:
cat outputs/hello.txt

Output:

```
hello neurodesktop
```

## Section 4: View History and Rerun

Install the dataset elsewhere (as if you were someone else):

In [ ]:
cd ~
datalad install -s ~/neurodesktop-storage/SomeProject
cd SomeProject

View the recorded provenance:

In [ ]:
git log outputs/hello.txt

The log shows the full `datalad run` record including the command, inputs, and outputs.

Re-run the analysis from the recorded commit:

In [ ]:
datalad rerun 52cf  # replace with your actual commit hash prefix

> **Git commit IDs** — Each Git commit has a unique machine-generated ID. Usually 4 characters are enough to uniquely identify a commit (e.g., `52cf`). Your commit ID will be different — check `git log` to find yours.

## Summary

Created a YODA-structured DataLad dataset, wrote a Julia script, used `datalad run` to record the command and its outputs, and used `datalad rerun` to reproduce the analysis from commit history.

## See Also

- DataLad tutorial (publishing/accessing datasets)
- osfclient tutorial